# H4: Discussion Quality and Market Reaction

This notebook tests whether the CAR response to small-shareholder victories is larger when the preceding discussion is higher quality. It follows the existing `scripts/reg` workflow: import the regression panel, estimate in Stata with `reghdfe`, and export LaTeX tables with `esttab`.

In [1]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      StataNow 19.5
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2025 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 2-core network, expiring 31 Oct 2026
Serial number: 501909305069
  Licensed to: Yichen Luo
               UCL

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


In [2]:
%%stata

clear all
set more off
set varabbrev off
version 18

* Paths
global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-governance/processed_data"
global TABLES "tables"
cap mkdir "$TABLES"



. 
. clear all

. set more off

. set varabbrev off

. version 18

. 
. * Paths
. global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-gove
> rnance/processed_data"

. global TABLES "tables"

. cap mkdir "$TABLES"

. 


In [3]:
%%stata

* Load proposal-level panel built by scripts/reg/reg_small.py
import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear

* Parse date and fixed effects
capture drop day
capture drop year
gen day = date(date, "YMD")
format day %td
gen year = year(day)

* H4 sample and variables
capture drop quality_defined
capture drop small_quality
capture drop nonsmall_win
capture drop sw_x_qdiff
capture drop sw_x_qfull
capture drop nonsw_x_qdiff
capture drop nonsw_x_qfull

gen quality_defined = post_number >= 2 & !missing(concensus_diff) & !missing(concensus_full)
gen small_quality = non_whale_victory_vn == 1 & quality_defined
gen nonsmall_win = 1 - non_whale_victory_vn

gen sw_x_qdiff = non_whale_victory_vn * concensus_diff
gen sw_x_qfull = non_whale_victory_vn * concensus_full
gen nonsw_x_qdiff = nonsmall_win * concensus_diff
gen nonsw_x_qfull = nonsmall_win * concensus_full

* Labels
label var car_created             "\${\it CAR}^{\it Create}_{i}\$"
label var car_end                 "\${\it CAR}^{\it End}_{i}\$"
label var non_whale_victory_vn    "\${\it Small Win}_{i}\$"
label var nonsmall_win            "\${\it NonSmallWin}_{i}\$"
label var concensus_diff          "\${\it \Delta Concensus}_{i}\$"
label var concensus_full          "\${\it Concensus}^{\it full}_{i}\$"
label var sw_x_qdiff              "\${\it Small Win}_{i} \times {\it \Delta Concensus}_{i}\$"
label var sw_x_qfull              "\${\it Small Win}_{i} \times {\it Concensus}^{\it full}_{i}\$"
label var nonsw_x_qdiff           "\${\it NonSmallWin}_{i} \times {\it \Delta Concensus}_{i}\$"
label var nonsw_x_qfull           "\${\it NonSmallWin}_{i} \times {\it Concensus}^{\it full}_{i}\$"



. 
. * Load proposal-level panel built by scripts/reg/reg_small.py
. import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear
(encoding automatically selected: ISO-8859-1)
(66 vars, 2,830 obs)

. 
. * Parse date and fixed effects
. capture drop day

. capture drop year

. gen day = date(date, "YMD")

. format day %td

. gen year = year(day)

. 
. * H4 sample and variables
. capture drop quality_defined

. capture drop small_quality

. capture drop nonsmall_win

. capture drop sw_x_qdiff

. capture drop sw_x_qfull

. capture drop nonsw_x_qdiff

. capture drop nonsw_x_qfull

. 
. gen quality_defined = post_number >= 2 & !missing(concensus_diff) & !missing(
> concensus_full)

. gen small_quality = non_whale_victory_vn == 1 & quality_defined

. gen nonsmall_win = 1 - non_whale_victory_vn
(161 missing values generated)

. 
. gen sw_x_qdiff = non_whale_victory_vn * concensus_diff
(2,274 missing values generated)

. gen sw_x_qfull = non_whale_victory_vn * concensus_full
(2,274 mis

In [4]:
%%stata

******************************************************
* FEASIBILITY CHECK - RUN FIRST
******************************************************

count if quality_defined
local n_quality = r(N)
count if quality_defined & !missing(car_created)
local n_car_created = r(N)
count if quality_defined & !missing(car_end)
local n_car_end = r(N)
count if small_quality
local n_small_quality = r(N)
count if small_quality & !missing(car_created)
local n_small_car_created = r(N)
count if small_quality & !missing(car_end)
local n_small_car_end = r(N)

display ""
display "H4 feasibility counts"
display "Proposals with >=2 discussion posts and quality defined: `n_quality'"
display "  of which non-missing CARCreate: `n_car_created'"
display "  of which non-missing CAREnd: `n_car_end'"
display "SmallWin = 1 and quality defined: `n_small_quality'"
display "SmallWin = 1, quality defined, and non-missing CARCreate: `n_small_car_created'"
display "SmallWin = 1, quality defined, and non-missing CAREnd: `n_small_car_end'"

capture file close h4feas
file open h4feas using "$TABLES/h4_feasibility.tex", write replace
file write h4feas "\begin{tabular}{lc}" _n
file write h4feas "\toprule" _n
file write h4feas "Cell & Count \\\\" _n
file write h4feas "\midrule" _n
file write h4feas "Proposals with $\geq 2$ discussion posts (quality defined) & `n_quality' \\\\" _n
file write h4feas "\quad of which non-missing $\mathrm{CARCreate}$ & `n_car_created' \\\\" _n
file write h4feas "\quad of which non-missing $\mathrm{CAREnd}$ & `n_car_end' \\\\" _n
file write h4feas "$\mathrm{SmallWin}_i=1$ and quality defined & `n_small_quality' \\\\" _n
file write h4feas "$\mathrm{SmallWin}_i=1$, quality defined, and non-missing $\mathrm{CARCreate}$ & `n_small_car_created' \\\\" _n
file write h4feas "$\mathrm{SmallWin}_i=1$, quality defined, and non-missing $\mathrm{CAREnd}$ & `n_small_car_end' \\\\" _n
file write h4feas "\bottomrule" _n
file write h4feas "\end{tabular}" _n
file close h4feas



. 
. ******************************************************
. * FEASIBILITY CHECK - RUN FIRST
. ******************************************************
. 
. count if quality_defined
  232

. local n_quality = r(N)

. count if quality_defined & !missing(car_created)
  158

. local n_car_created = r(N)

. count if quality_defined & !missing(car_end)
  158

. local n_car_end = r(N)

. count if small_quality
  21

. local n_small_quality = r(N)

. count if small_quality & !missing(car_created)
  21

. local n_small_car_created = r(N)

. count if small_quality & !missing(car_end)
  21

. local n_small_car_end = r(N)

. 
. display ""


. display "H4 feasibility counts"
H4 feasibility counts

. display "Proposals with >=2 discussion posts and quality defined: `n_quality'
> "
Proposals with >=2 discussion posts and quality defined: 232

. display "  of which non-missing CARCreate: `n_car_created'"
  of which non-missing CARCreate: 158

. display "  of which non-missing CAREnd: `n_car_end'"
  o

In [5]:
%%stata

******************************************************
* MAIN H4 INTERACTION REGRESSIONS
******************************************************

eststo clear

reghdfe car_created non_whale_victory_vn sw_x_qdiff concensus_diff if quality_defined, absorb(year categories) vce(cluster categories)
eststo qdiff_created
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end non_whale_victory_vn sw_x_qdiff concensus_diff if quality_defined, absorb(year categories) vce(cluster categories)
eststo qdiff_end
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_created non_whale_victory_vn sw_x_qfull concensus_full if quality_defined, absorb(year categories) vce(cluster categories)
eststo qfull_created
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end non_whale_victory_vn sw_x_qfull concensus_full if quality_defined, absorb(year categories) vce(cluster categories)
eststo qfull_end
estadd local fe_proposal "Y"
estadd local fe_time "Y"

esttab                                                           ///
    qdiff_created qdiff_end qfull_created qfull_end              ///
    using "$TABLES/reg_h4_interaction.tex", replace              ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs nomtitles                                   ///
    b(%9.3f) se(%9.2f)                                           ///
    keep(non_whale_victory_vn sw_x_qdiff sw_x_qfull concensus_diff concensus_full) ///
    order(non_whale_victory_vn sw_x_qdiff sw_x_qfull concensus_diff concensus_full) ///
    mgroups("Quality $=\Delta$ Consensus" "Quality $=$ Consensus level", ///
            span pattern(1 0 1 0) prefix(\multicolumn{@span}{c}{) suffix(}) ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    posthead(\cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{\${\it CAR}^{\it Create}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it End}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it Create}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it End}_{i}\$} \\ \cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)} \\ \midrule) ///
    substitute("\_" "_")                                         ///
    stats(fe_proposal fe_time N r2_a,                               ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
        labels("Category FE" "Year FE" "Observations" "Adjusted R²"))



. 
. ******************************************************
. * MAIN H4 INTERACTION REGRESSIONS
. ******************************************************
. 
. eststo clear

. 
. reghdfe car_created non_whale_victory_vn sw_x_qdiff concensus_diff if quality
> _defined, absorb(year categories) vce(cluster categories)
(MWFE estimator converged in 5 iterations)

HDFE Linear regression                            Number of obs   =        152
Absorbing 2 HDFE groups                           F(   3,      3) =     252.18
Statistics robust to heteroskedasticity           Prob > F        =     0.0004
                                                  R-squared       =     0.0887
                                                  Adj R-squared   =     0.0241
                                                  Within R-sq.    =     0.0109
Number of clusters (categories) =          4      Root MSE        =     0.1422

                             (Std. err. adjusted for 4 clusters in categories)
-------

In [6]:
%%stata

******************************************************
* BACKUP MEDIAN-SPLIT REGRESSIONS
******************************************************

encode gecko_id, gen(token)
encode space, gen(dao)

capture drop high_qfull
summ concensus_full if quality_defined, detail
gen high_qfull = concensus_full > r(p50) if quality_defined & !missing(concensus_full)

* Winsorize CAR outcomes before estimating the split-sample regressions
capture drop car_created_split
capture drop car_end_split
gen car_created_split = car_created
gen car_end_split = car_end
foreach v in car_created_split car_end_split {
    quietly summ `v' if quality_defined, detail
    replace `v' = r(p1) if `v' < r(p1) & !missing(`v')
    replace `v' = r(p99) if `v' > r(p99) & !missing(`v')
}
label var car_created_split "\${\it CAR}^{\it Create}_{i}\$"
label var car_end_split "\${\it CAR}^{\it End}_{i}\$"

eststo clear

reghdfe car_created_split non_whale_victory_vn if quality_defined & high_qfull == 0, absorb(year categories) vce(cluster categories)
eststo created_low
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_created_split non_whale_victory_vn if quality_defined & high_qfull == 1, absorb(year categories) vce(cluster categories)
eststo created_high
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end_split non_whale_victory_vn if quality_defined & high_qfull == 0, absorb(year categories) vce(cluster categories)
eststo end_low
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end_split non_whale_victory_vn if quality_defined & high_qfull == 1, absorb(year categories) vce(cluster categories)
eststo end_high
estadd local fe_proposal "Y"
estadd local fe_time "Y"

esttab                                                           ///
    created_low created_high end_low end_high                    ///
    using "$TABLES/reg_h4_split.tex", replace                    ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs nomtitles                                   ///
    b(%9.3f) se(%9.2f)                                           ///
    keep(non_whale_victory_vn)                                   ///
    mgroups("\${\it CAR}^{\it Create}_{i}\$" "\${\it CAR}^{\it End}_{i}\$", ///
            span pattern(1 0 1 0) prefix(\multicolumn{@span}{c}{) suffix(}) ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    posthead(\cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{Low quality} & \multicolumn{1}{c}{High quality} & \multicolumn{1}{c}{Low quality} & \multicolumn{1}{c}{High quality} \\ \cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)} \\ \midrule) ///
    substitute("\_" "_")                                         ///
    stats(fe_proposal fe_time N r2_a,                               ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
        labels("Category FE" "Year FE" "Observations" "Adjusted R²"))



. 
. ******************************************************
. * BACKUP MEDIAN-SPLIT REGRESSIONS
. ******************************************************
. 
. encode gecko_id, gen(token)

. encode space, gen(dao)

. 
. capture drop high_qfull

. summ concensus_full if quality_defined, detail

              ${\it Concensus}^{\it full}_{i}$
-------------------------------------------------------------
      Percentiles      Smallest
 1%     .0031827       .0006622
 5%      .008605       .0022227
10%     .0229986       .0031827       Obs                 232
25%      .257091       .0041013       Sum of wgt.         232

50%     .6923075                      Mean           .6041927
                        Largest       Std. dev.      .3744334
75%     .9763304       .9997901
90%     .9979337       .9998584       Variance       .1402003
95%     .9991828       .9998797       Skewness      -.4167301
99%     .9998584       .9998909       Kurtosis       1.597724

. gen high_qfull = concensus_full

In [7]:
%%stata

******************************************************
* PLACEBO: NON-SMALL-WIN INTERACTION
******************************************************

eststo clear

reghdfe car_created nonsmall_win nonsw_x_qdiff concensus_diff if quality_defined, absorb(year categories) vce(cluster categories)
eststo qdiff_created
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end nonsmall_win nonsw_x_qdiff concensus_diff if quality_defined, absorb(year categories) vce(cluster categories)
eststo qdiff_end
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_created nonsmall_win nonsw_x_qfull concensus_full if quality_defined, absorb(year categories) vce(cluster categories)
eststo qfull_created
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end nonsmall_win nonsw_x_qfull concensus_full if quality_defined, absorb(year categories) vce(cluster categories)
eststo qfull_end
estadd local fe_proposal "Y"
estadd local fe_time "Y"

esttab                                                           ///
    qdiff_created qdiff_end qfull_created qfull_end              ///
    using "$TABLES/reg_h4_interaction_placebo.tex", replace      ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs nomtitles                                   ///
    b(%9.3f) se(%9.2f)                                           ///
    keep(nonsmall_win nonsw_x_qdiff nonsw_x_qfull concensus_diff concensus_full) ///
    order(nonsmall_win nonsw_x_qdiff nonsw_x_qfull concensus_diff concensus_full) ///
    mgroups("Quality $=\Delta$ Consensus" "Quality $=$ Consensus level", ///
            span pattern(1 0 1 0) prefix(\multicolumn{@span}{c}{) suffix(}) ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    posthead(\cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{\${\it CAR}^{\it Create}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it End}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it Create}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it End}_{i}\$} \\ \cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)} \\ \midrule) ///
    substitute("\_" "_")                                         ///
    stats(fe_proposal fe_time N r2_a,                               ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
        labels("Category FE" "Year FE" "Observations" "Adjusted R²"))



. 
. ******************************************************
. * PLACEBO: NON-SMALL-WIN INTERACTION
. ******************************************************
. 
. eststo clear

. 
. reghdfe car_created nonsmall_win nonsw_x_qdiff concensus_diff if quality_defi
> ned, absorb(year categories) vce(cluster categories)
(MWFE estimator converged in 5 iterations)

HDFE Linear regression                            Number of obs   =        152
Absorbing 2 HDFE groups                           F(   3,      3) =     252.18
Statistics robust to heteroskedasticity           Prob > F        =     0.0004
                                                  R-squared       =     0.0887
                                                  Adj R-squared   =     0.0241
                                                  Within R-sq.    =     0.0109
Number of clusters (categories) =          4      Root MSE        =     0.1422

                             (Std. err. adjusted for 4 clusters in categories)
---------

In [8]:
%%stata

******************************************************
* WINSORIZED ROBUSTNESS CHECK
******************************************************

preserve
foreach v in concensus_diff concensus_full {
    quietly summ `v', detail
    replace `v' = r(p99) if `v' > r(p99) & !missing(`v')
}

capture drop sw_x_qdiff_w sw_x_qfull_w
gen sw_x_qdiff_w = non_whale_victory_vn * concensus_diff
gen sw_x_qfull_w = non_whale_victory_vn * concensus_full
label var sw_x_qdiff_w "\${\it Small Win}_{i} \times {\it \Delta Concensus}_{i}\$"
label var sw_x_qfull_w "\${\it Small Win}_{i} \times {\it Concensus}^{\it full}_{i}\$"

eststo clear

reghdfe car_created non_whale_victory_vn sw_x_qdiff_w concensus_diff if quality_defined, absorb(year categories) vce(cluster categories)
eststo qdiff_created
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end non_whale_victory_vn sw_x_qdiff_w concensus_diff if quality_defined, absorb(year categories) vce(cluster categories)
eststo qdiff_end
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_created non_whale_victory_vn sw_x_qfull_w concensus_full if quality_defined, absorb(year categories) vce(cluster categories)
eststo qfull_created
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end non_whale_victory_vn sw_x_qfull_w concensus_full if quality_defined, absorb(year categories) vce(cluster categories)
eststo qfull_end
estadd local fe_proposal "Y"
estadd local fe_time "Y"

esttab                                                           ///
    qdiff_created qdiff_end qfull_created qfull_end              ///
    using "$TABLES/reg_h4_interaction_winsorized.tex", replace   ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs nomtitles                                   ///
    b(%9.3f) se(%9.2f)                                           ///
    keep(non_whale_victory_vn sw_x_qdiff_w sw_x_qfull_w concensus_diff concensus_full) ///
    order(non_whale_victory_vn sw_x_qdiff_w sw_x_qfull_w concensus_diff concensus_full) ///
    mgroups("Quality $=\Delta$ Consensus" "Quality $=$ Consensus level", ///
            span pattern(1 0 1 0) prefix(\multicolumn{@span}{c}{) suffix(}) ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    posthead(\cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{\${\it CAR}^{\it Create}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it End}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it Create}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it End}_{i}\$} \\ \cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)} \\ \midrule) ///
    substitute("\_" "_")                                         ///
    stats(fe_proposal fe_time N r2_a,                               ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
        labels("Category FE" "Year FE" "Observations" "Adjusted R²"))
restore



. 
. ******************************************************
. * WINSORIZED ROBUSTNESS CHECK
. ******************************************************
. 
. preserve

. foreach v in concensus_diff concensus_full {
  2.     quietly summ `v', detail
  3.     replace `v' = r(p99) if `v' > r(p99) & !missing(`v')
  4. }
(5 real changes made)
(5 real changes made)

. 
. capture drop sw_x_qdiff_w sw_x_qfull_w

. gen sw_x_qdiff_w = non_whale_victory_vn * concensus_diff
(2,274 missing values generated)

. gen sw_x_qfull_w = non_whale_victory_vn * concensus_full
(2,274 missing values generated)

. label var sw_x_qdiff_w "\${\it Small Win}_{i} \times {\it \Delta Concensus}_{
> i}\$"

. label var sw_x_qfull_w "\${\it Small Win}_{i} \times {\it Concensus}^{\it ful
> l}_{i}\$"

. 
. eststo clear

. 
. reghdfe car_created non_whale_victory_vn sw_x_qdiff_w concensus_diff if quali
> ty_defined, absorb(year categories) vce(cluster categories)
(MWFE estimator converged in 5 iterations)

HDFE Linear regress